# ETL — Minnie65: Cell Features

Writes the CSM dendrite-ultrastructure cohort `DataSet` (`minnie65_v1300_csm_cluster`), its `DataItemDataSetAssociation` links, `CellFeatureDefinition` rows, `CellFeatureSet` rows, wide-form feature parquet tables, and `CellFeatureMatrix` pointer rows for two feature sets. Each feature-set section is independently idempotent. Prerequisite: `etl_minnie_01_dataset_dataitem.ipynb`.

In [1]:
import os
from pathlib import Path

import caveclient
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import standard_transform
from deltalake import write_deltalake

from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    DataItemDataSetAssociation,
    DataSet,
    Modality,
    Unit,
)
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


/opt/conda/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.2) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [2]:
FEATURES_PARQUET  = "/data/minnie1412/minnie_features.parquet"
FEATURES_CSV      = "/data/minnie1412/minnie_cell_features.csv"
OUTPUT_ROOT       = output_root()
PROJECT_ID        = "minnie65"
COHORT_DATASET_ID = "minnie65_v1300_csm_cluster"
FSI_CSM           = "csm_cluster_features"
FSI_STD           = "minnie65_std_transform_coordinates"
CAVE_DATASTACK    = "minnie65_phase3_v1"
CAVE_VERSION      = 1300
CAVE_VIEW         = "nucleus_detection_lookup_v1"

print(f"OUTPUT_ROOT       : {OUTPUT_ROOT}")
print(f"PROJECT_ID        : {PROJECT_ID}")
print(f"COHORT_DATASET_ID : {COHORT_DATASET_ID}")
print(f"FSI_CSM           : {FSI_CSM}")
print(f"FSI_STD           : {FSI_STD}")

OUTPUT_ROOT       : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID        : minnie65
COHORT_DATASET_ID : minnie65_v1300_csm_cluster
FSI_CSM           : csm_cluster_features
FSI_STD           : minnie65_std_transform_coordinates


## Prerequisite check

In [3]:
prereq = (
    pl.read_delta(OUTPUT_ROOT + "dataset/")
    .filter(pl.col("id") == "minnie65_v1300_nuclei")
)
assert prereq.shape[0] == 1, "etl_minnie_01 must be run first — minnie65_v1300_nuclei DataSet not found"
print("Prerequisite OK:", prereq["id"][0])

Prerequisite OK: minnie65_v1300_nuclei


## Load inputs

In [4]:
feat_df   = pd.read_parquet(FEATURES_PARQUET)
feat_meta = pd.read_csv(FEATURES_CSV)

# 4 duplicate ids exist in the source parquet; drop them to keep the feature
# table cell-indexed (one row per nucleus id).
n_before = len(feat_df)
feat_df = feat_df.drop_duplicates(subset="id", keep="first")
print(f"Dropped {n_before - len(feat_df)} duplicate id row(s): {n_before} → {len(feat_df)}")

print("Features parquet shape:", feat_df.shape)
print("Feature metadata CSV shape:", feat_meta.shape)
display(feat_df.head(3))
display(feat_meta.head(3))

Dropped 4 duplicate id row(s): 35787 → 35783
Features parquet shape: (35783, 112)
Feature metadata CSV shape: (82, 6)


,root_id,soma_depth,tip_len_dist_dendrite_p75,tip_tort_dendrite_p75,num_syn_dendrite,num_syn_soma,path_length_dendrite,radial_extent_dendrite,syn_dist_distribution_dendrite_p50,syn_size_distribution_soma_p50,...,fraction_nuclear_folding,nucleus_to_soma_ratio,soma_volume_um,soma_area_um,soma_to_nucleus_center_dist,soma_area_to_volume_ratio,soma_synapse_density_um,valence,id,pt_root_id_y
0,864691136740606812,115.504525,147.846945,1.264484,6768,117,4849.9004,227.368087,88.470299,3532.0,...,0.017687,0.309179,1112.999910,1202.533434,537.425091,1.080443,0.158170,exc,485509,864691136740606812
1,864691136210678204,355.431707,152.745301,1.301633,2036,45,2394.6440,115.675705,74.979199,4564.0,...,0.123807,0.397382,640.537711,934.029388,361.333208,1.458196,0.119974,exc,263203,864691136210678204
2,864691134965388575,237.838347,153.671766,1.350950,4380,124,4331.2370,107.554109,86.696932,4408.0,...,0.124294,0.366174,923.129563,1017.687965,365.622018,1.102432,0.189111,exc,456177,864691134965388575


,id,description,unit,data_type,range_min,range_max
0,nucleus_volume_um,Nucleus volume,MICRONS_CUBED,<f4,0.0,NaN
1,nucleus_area_um,Nucleus surface area,MICRONS_SQUARE,<f4,0.0,NaN
2,nuclear_area_to_volume_ratio,Nucleus surface area to volume ratio,MICRONS_INVERSE,<f4,0.0,NaN


## Write cohort `DataSet` + `DataItemDataSetAssociation`

In [12]:
cohort_ds = DataSet(
    id=COHORT_DATASET_ID,
    name="Minnie65 v1300 CSM dendrite ultrastructure cohort",
    modality=Modality.ELECTRON_MICROSCOPY.value,
    project_id=PROJECT_ID,
)
result = write_models([cohort_ds], output_root=OUTPUT_ROOT)

In [13]:
cell_ids = feat_df["id"].astype(str).tolist()
associations = [
    DataItemDataSetAssociation(dataitem_id=cid, dataset_id=COHORT_DATASET_ID, project_id=PROJECT_ID)
    for cid in cell_ids
]
result = write_models(associations, output_root=OUTPUT_ROOT)

In [14]:
# Verification
ds_v = pl.read_delta(OUTPUT_ROOT + "dataset/").filter(pl.col("id") == COHORT_DATASET_ID)
assoc_v = (pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
           .filter(pl.col("dataset_id") == COHORT_DATASET_ID))
print("DataSet:", ds_v.shape); print(ds_v.head())
print("Associations:", assoc_v.shape)
assert ds_v.shape[0] == 1
assert assoc_v.shape[0] == len(cell_ids)

DataSet: (1, 5)
shape: (1, 5)
┌────────────────────────────┬────────────────────┬─────────────┬─────────────────────┬────────────┐
│ id                         ┆ name               ┆ publication ┆ modality            ┆ project_id │
│ ---                        ┆ ---                ┆ ---         ┆ ---                 ┆ ---        │
│ str                        ┆ str                ┆ str         ┆ str                 ┆ str        │
╞════════════════════════════╪════════════════════╪═════════════╪═════════════════════╪════════════╡
│ minnie65_v1300_csm_cluster ┆ Minnie65 v1300 CSM ┆ null        ┆ ELECTRON_MICROSCOPY ┆ minnie65   │
│                            ┆ dendrite ul…       ┆             ┆                     ┆            │
└────────────────────────────┴────────────────────┴─────────────┴─────────────────────┴────────────┘
Associations: (35783, 3)


---
## Feature set A: `csm_cluster_features`

Each write predicates on both `project_id` and `feature_set_id`, so this section and the STD section are fully independent and either can be re-run without affecting the other.

In [15]:
# Build CSM CellFeatureDefinitions from CSV
csm_fds = []
for _, row in feat_meta.iterrows():
    kwargs = dict(
        id=str(row["id"]),
        description=str(row["description"]),
        unit=str(row["unit"]),
        data_type=str(row["data_type"]),
        project_id=PROJECT_ID,
        feature_set_id=FSI_CSM,
    )
    if pd.notna(row["range_min"]):
        kwargs["range_min"] = float(row["range_min"])
    if pd.notna(row["range_max"]):
        kwargs["range_max"] = float(row["range_max"])
    csm_fds.append(CellFeatureDefinition(**kwargs))
result = write_models(csm_fds, output_root=OUTPUT_ROOT)
print(f"CellFeatureDefinition written: {result.rows_written} rows")

CellFeatureDefinition written: 82 rows


In [16]:
# Verification
cfd_csm_v = (pl.read_delta(OUTPUT_ROOT + "cellfeaturedefinition/")
             .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FSI_CSM)))
print(cfd_csm_v.shape); print(cfd_csm_v.head(3))
assert cfd_csm_v.shape[0] == len(csm_fds)
assert cfd_csm_v["id"].n_unique() == len(csm_fds)

(82, 8)
shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ descriptio ┆ unit       ┆ data_type ┆ range_min ┆ range_max ┆ project_i ┆ feature_s │
│ ---        ┆ n          ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ d         ┆ et_id     │
│ str        ┆ ---        ┆ str        ┆ str       ┆ f64       ┆ f64       ┆ ---       ┆ ---       │
│            ┆ str        ┆            ┆           ┆           ┆           ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ nucleus_vo ┆ Nucleus    ┆ MICRONS_CU ┆ <f4       ┆ 0.0       ┆ null      ┆ minnie65  ┆ csm_clust │
│ lume_um    ┆ volume     ┆ BED        ┆           ┆           ┆           ┆           ┆ er_featur │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ es        │
│ nucleus_ar ┆ Nucleus    ┆ MICRONS_SQ ┆ <f4       ┆ 0.0       ┆ null

In [17]:
# Build and write CSM CellFeatureSet
cfs_csm = CellFeatureSet(
    id=FSI_CSM,
    description=(
        "Cell features used for clustering in the Allen Institute's large scale EM projects. "
        "Contains features from Elabbady et al 2025, Schneider-Mizell et al 2025, and "
        "spine detection features from Ben Pedigo. "
        "Feature set developed by Casey Schneider-Mizell. "
        "Takes a synapse-centric morphological approach with features "
        "describing how synapse densities are distributed across the dendritic arbors."
    ),
    feature_definition_ids=[fd.id for fd in csm_fds],
    extraction_method="Aggregated and computed via https://github.com/AllenInstitute/em_skeleton_feature_extraction.",
    project_id=PROJECT_ID,
)
result = write_models([cfs_csm], output_root=OUTPUT_ROOT)
print(f"CellFeatureSet written: {result.rows_written} rows")

CellFeatureSet written: 1 rows


In [18]:
# Verification
cfs_csm_v = (pl.read_delta(OUTPUT_ROOT + "cellfeatureset/")
             .filter(pl.col("id") == FSI_CSM))
print(cfs_csm_v.shape); print(cfs_csm_v)
assert cfs_csm_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌─────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────┐
│ id                  ┆ description         ┆ feature_definition ┆ extraction_method  ┆ project_id │
│ ---                 ┆ ---                 ┆ _ids               ┆ ---                ┆ ---        │
│ str                 ┆ str                 ┆ ---                ┆ str                ┆ str        │
│                     ┆                     ┆ list[str]          ┆                    ┆            │
╞═════════════════════╪═════════════════════╪════════════════════╪════════════════════╪════════════╡
│ csm_cluster_feature ┆ Cell features used  ┆ ["nucleus_volume_u ┆ Aggregated and     ┆ minnie65   │
│ s                   ┆ for cluster…        ┆ m", "nucleus…      ┆ computed via ht…   ┆            │
└─────────────────────┴─────────────────────┴────────────────────┴────────────────────┴────────────┘


In [20]:
# Build and write wide-form CSM feature parquet
csm_feat_df = feat_df.drop(columns=["root_id", "pt_root_id_y", "valence"], errors="ignore").copy()
csm_feat_df["id"]             = feat_df["id"].astype(str)
csm_feat_df["project_id"]     = PROJECT_ID
csm_feat_df["feature_set_id"] = FSI_CSM

for fd in csm_fds:
    col = fd.id
    if col not in csm_feat_df.columns:
        continue
    if fd.data_type[1] == "f":
        csm_feat_df[col] = csm_feat_df[col].astype("float32")
    elif fd.data_type[1] == "i":
        csm_feat_df[col] = csm_feat_df[col].astype("int32")

schema_csm_wide = build_cell_feature_matrix_schema(cfs_csm, csm_fds, cell_index_column="id")
table_csm_wide  = pa.Table.from_pandas(csm_feat_df, schema=schema_csm_wide, preserve_index=False)

write_deltalake(
    OUTPUT_ROOT + f"cellfeatures/{FSI_CSM}/", table_csm_wide,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Wide csm_cluster_features written:", table_csm_wide.shape)

Wide csm_cluster_features written: (35783, 85)


In [21]:
# Verification
csm_v = pl.read_delta(OUTPUT_ROOT + f"cellfeatures/{FSI_CSM}/").filter(pl.col("project_id") == PROJECT_ID)
print(csm_v.shape); print(csm_v.head(3))
assert csm_v.shape[0] == len(cell_ids)
assert csm_v["id"].n_unique() == len(cell_ids)

(35783, 85)
shape: (3, 85)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id     ┆ nucleus_vo ┆ nucleus_ar ┆ nuclear_a ┆ … ┆ ego_count ┆ ego_count ┆ project_i ┆ feature_s │
│ ---    ┆ lume_um    ┆ ea_um      ┆ rea_to_vo ┆   ┆ _pca1     ┆ _pca2     ┆ d         ┆ et_id     │
│ str    ┆ ---        ┆ ---        ┆ lume_rati ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│        ┆ f32        ┆ f32        ┆ o         ┆   ┆ f32       ┆ f32       ┆ str       ┆ str       │
│        ┆            ┆            ┆ ---       ┆   ┆           ┆           ┆           ┆           │
│        ┆            ┆            ┆ f32       ┆   ┆           ┆           ┆           ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 485509 ┆ 344.116638 ┆ 269.338379 ┆ 0.782695  ┆ … ┆ 0.805681  ┆ -0.739578 ┆ minnie65  ┆ csm_clust │
│        ┆            ┆            ┆           ┆   ┆           ┆

In [22]:
# Build and write CSM CellFeatureMatrix pointer
output_abs = Path(OUTPUT_ROOT).resolve()
cfm_csm = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FSI_CSM}",
    feature_set_id=FSI_CSM,
    parquet_path=f"file://{output_abs}/cellfeatures/{FSI_CSM}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
result = write_models([cfm_csm], output_root=OUTPUT_ROOT)
print(f"CellFeatureMatrix written: {result.rows_written} rows")

CellFeatureMatrix written: 1 rows


In [23]:
# Verification
cfm_csm_v = (pl.read_delta(OUTPUT_ROOT + "cellfeaturematrix/")
             .filter(pl.col("feature_set_id") == FSI_CSM))
print(cfm_csm_v.shape); print(cfm_csm_v)
assert cfm_csm_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌─────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬────────────┐
│ id                  ┆ feature_set_id      ┆ parquet_path        ┆ cell_index_column ┆ project_id │
│ ---                 ┆ ---                 ┆ ---                 ┆ ---               ┆ ---        │
│ str                 ┆ str                 ┆ str                 ┆ str               ┆ str        │
╞═════════════════════╪═════════════════════╪═════════════════════╪═══════════════════╪════════════╡
│ minnie65_csm_cluste ┆ csm_cluster_feature ┆ file:///scratch/em_ ┆ id                ┆ minnie65   │
│ r_features          ┆ s                   ┆ patchseq_wn…        ┆                   ┆            │
└─────────────────────┴─────────────────────┴─────────────────────┴───────────────────┴────────────┘


---
## Feature set B: `minnie65_std_transform_coordinates`

Query CAVE, apply `standard_transform`, write coordinate feature definitions, set, wide parquet, and matrix. All writes predicate on `feature_set_id = FSI_STD` — CSM rows are untouched.

In [24]:
# Query CAVE → apply standard_transform → soma coordinates in microns
client = caveclient.CAVEclient(CAVE_DATASTACK, auth_token=os.environ["CUSTOM_KEY"])
client.materialize.version = CAVE_VERSION

nuc_df = client.materialize.query_view(CAVE_VIEW)
nuc_df = nuc_df.query("pt_root_id != 0")

tform = standard_transform.minnie_transform_vx()
xt = tform.apply_dataframe("pt_position", nuc_df, projection="x")
yt = tform.apply_dataframe("pt_position", nuc_df, projection="y")
zt = tform.apply_dataframe("pt_position", nuc_df, projection="z")

cortical_coord_df = pd.DataFrame({
    "id":                nuc_df["id"].astype(str).values,
    "x_medial-lateral": np.array(xt, dtype=np.float32) / 1000,
    "y_dorsal-ventral": np.array(yt, dtype=np.float32) / 1000,
    "z_caudal-rostral": np.array(zt, dtype=np.float32) / 1000,
})

print("cortical_coord_df shape:", cortical_coord_df.shape)
cortical_coord_df.head(3)

cortical_coord_df shape: (133969, 4)


,id,x_medial-lateral,y_dorsal-ventral,z_caudal-rostral
0,373879,0.828190,0.638554,0.78372
1,201858,0.510691,0.505672,1.05068
2,600774,1.255059,0.821799,0.77768


In [25]:
# Build and write STD CellFeatureDefinitions
std_fds = [
    CellFeatureDefinition(
        id="x_medial-lateral",
        description=(
            "The x coordinate in minnie65 after applying standard_transform package. "
            "Lower is more toward midline of brain (medial), higher is more lateral on the right side of brain."
        ),
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FSI_STD,
    ),
    CellFeatureDefinition(
        id="y_dorsal-ventral",
        description=(
            "The y coordinate in minnie65 after applying standard_transform package. "
            "Lower is more dorsal, with 0 being the surface of the brain (pia), higher is deeper toward white matter."
        ),
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FSI_STD,
    ),
    CellFeatureDefinition(
        id="z_caudal-rostral",
        description=(
            "The z coordinate in minnie65 after applying standard_transform package. "
            "Lower is further back in the brain (caudal), higher is toward the nose (rostral)."
        ),
        unit=Unit.MICRONS_LENGTH.value,
        data_type="<f4",
        project_id=PROJECT_ID,
        feature_set_id=FSI_STD,
    ),
]
result = write_models(std_fds, output_root=OUTPUT_ROOT)
print(f"CellFeatureDefinition written: {result.rows_written} rows")

CellFeatureDefinition written: 3 rows


In [26]:
# Verification
cfd_std_v = (pl.read_delta(OUTPUT_ROOT + "cellfeaturedefinition/")
             .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FSI_STD)))
print(cfd_std_v.shape); print(cfd_std_v)
assert cfd_std_v.shape[0] == 3
assert set(cfd_std_v["id"].to_list()) == {"x_medial-lateral", "y_dorsal-ventral", "z_caudal-rostral"}

(3, 8)
shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ descriptio ┆ unit       ┆ data_type ┆ range_min ┆ range_max ┆ project_i ┆ feature_s │
│ ---        ┆ n          ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ d         ┆ et_id     │
│ str        ┆ ---        ┆ str        ┆ str       ┆ f64       ┆ f64       ┆ ---       ┆ ---       │
│            ┆ str        ┆            ┆           ┆           ┆           ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ x_medial-l ┆ The x      ┆ MICRONS_LE ┆ <f4       ┆ null      ┆ null      ┆ minnie65  ┆ minnie65_ │
│ ateral     ┆ coordinate ┆ NGTH       ┆           ┆           ┆           ┆           ┆ std_trans │
│            ┆ in         ┆            ┆           ┆           ┆           ┆           ┆ form_coor │
│            ┆ minnie65   ┆            ┆           ┆           ┆      

In [27]:
# Build and write STD CellFeatureSet
cfs_std = CellFeatureSet(
    id=FSI_STD,
    description=(
        "The coordinates of cell somas after applying the standard_transform package, "
        "which levels the dataset and places y=0 at the surface of the brain (pia)."
    ),
    feature_definition_ids=[fd.id for fd in std_fds],
    extraction_method="Applied standard_transform.minnie_transform_vx() to nucleus_detection_lookup_v1 (pt_root_id != 0). Divided nm output by 1000 to convert to microns.",
    project_id=PROJECT_ID,
)
result = write_models([cfs_std], output_root=OUTPUT_ROOT)
print(f"CellFeatureSet written: {result.rows_written} rows")

CellFeatureSet written: 1 rows


In [28]:
# Verification
cfs_std_v = (pl.read_delta(OUTPUT_ROOT + "cellfeatureset/")
             .filter(pl.col("id") == FSI_STD))
print(cfs_std_v.shape); print(cfs_std_v)
assert cfs_std_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌─────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────┐
│ id                  ┆ description         ┆ feature_definition ┆ extraction_method  ┆ project_id │
│ ---                 ┆ ---                 ┆ _ids               ┆ ---                ┆ ---        │
│ str                 ┆ str                 ┆ ---                ┆ str                ┆ str        │
│                     ┆                     ┆ list[str]          ┆                    ┆            │
╞═════════════════════╪═════════════════════╪════════════════════╪════════════════════╪════════════╡
│ minnie65_std_transf ┆ The coordinates of  ┆ ["x_medial-lateral ┆ Applied standard_t ┆ minnie65   │
│ orm_coordin…        ┆ cell somas …        ┆ ", "y_dorsal…      ┆ ransform.min…      ┆            │
└─────────────────────┴─────────────────────┴────────────────────┴────────────────────┴────────────┘


In [29]:
# Build and write wide-form STD coordinate parquet
cortical_coord_df["project_id"]     = PROJECT_ID
cortical_coord_df["feature_set_id"] = FSI_STD

schema_std_wide = build_cell_feature_matrix_schema(cfs_std, std_fds, cell_index_column="id")
table_std_wide  = pa.Table.from_pandas(cortical_coord_df, schema=schema_std_wide, preserve_index=False)

write_deltalake(
    OUTPUT_ROOT + f"cellfeatures/{FSI_STD}/", table_std_wide,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Wide std_transform_coordinates written:", table_std_wide.shape)

Wide std_transform_coordinates written: (133969, 6)


In [30]:
# Verification
std_v = pl.read_delta(OUTPUT_ROOT + f"cellfeatures/{FSI_STD}/").filter(pl.col("project_id") == PROJECT_ID)
print(std_v.shape); print(std_v.head(3))
assert std_v.shape[0] == len(nuc_df)
assert std_v["id"].n_unique() == len(nuc_df)

(133969, 6)
shape: (3, 6)
┌────────┬──────────────────┬──────────────────┬──────────────────┬────────────┬───────────────────┐
│ id     ┆ x_medial-lateral ┆ y_dorsal-ventral ┆ z_caudal-rostral ┆ project_id ┆ feature_set_id    │
│ ---    ┆ ---              ┆ ---              ┆ ---              ┆ ---        ┆ ---               │
│ str    ┆ f32              ┆ f32              ┆ f32              ┆ str        ┆ str               │
╞════════╪══════════════════╪══════════════════╪══════════════════╪════════════╪═══════════════════╡
│ 373879 ┆ 0.82819          ┆ 0.638554         ┆ 0.78372          ┆ minnie65   ┆ minnie65_std_tran │
│        ┆                  ┆                  ┆                  ┆            ┆ sform_coordin…    │
│ 201858 ┆ 0.510691         ┆ 0.505672         ┆ 1.05068          ┆ minnie65   ┆ minnie65_std_tran │
│        ┆                  ┆                  ┆                  ┆            ┆ sform_coordin…    │
│ 600774 ┆ 1.255059         ┆ 0.821799         ┆ 0.77768         

In [31]:
# Build and write STD CellFeatureMatrix pointer
cfm_std = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FSI_STD}",
    feature_set_id=FSI_STD,
    parquet_path=f"file://{output_abs}/cellfeatures/{FSI_STD}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
result = write_models([cfm_std], output_root=OUTPUT_ROOT)
print(f"CellFeatureMatrix written: {result.rows_written} rows")

CellFeatureMatrix written: 1 rows


In [32]:
# Verification
cfm_std_v = (pl.read_delta(OUTPUT_ROOT + "cellfeaturematrix/")
             .filter(pl.col("feature_set_id") == FSI_STD))
print(cfm_std_v.shape); print(cfm_std_v)
assert cfm_std_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌─────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬────────────┐
│ id                  ┆ feature_set_id      ┆ parquet_path        ┆ cell_index_column ┆ project_id │
│ ---                 ┆ ---                 ┆ ---                 ┆ ---               ┆ ---        │
│ str                 ┆ str                 ┆ str                 ┆ str               ┆ str        │
╞═════════════════════╪═════════════════════╪═════════════════════╪═══════════════════╪════════════╡
│ minnie65_minnie65_s ┆ minnie65_std_transf ┆ file:///scratch/em_ ┆ id                ┆ minnie65   │
│ td_transfor…        ┆ orm_coordin…        ┆ patchseq_wn…        ┆                   ┆            │
└─────────────────────┴─────────────────────┴─────────────────────┴───────────────────┴────────────┘


## Summary

| Output path | Class | Rows |
|---|---|---|
| `dataset/` | `DataSet` | +1 (`minnie65_v1300_csm_cluster`) |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | one per cell in `minnie_features.parquet` |
| `cellfeaturedefinition/` | `CellFeatureDefinition` | 82 CSM + 3 coordinate = 85 |
| `cellfeatureset/` | `CellFeatureSet` | 2 |
| `cellfeatures/csm_cluster_features/` | wide parquet | `len(feat_df)` rows × 82 features |
| `cellfeatures/minnie65_std_transform_coordinates/` | wide parquet | `len(nuc_df)` rows × 3 coordinates |
| `cellfeaturematrix/` | `CellFeatureMatrix` | 2 |

Each feature-set section (A and B) predicates on both `project_id` and `feature_set_id`, so re-running section A does not affect section B's rows and vice versa.

**Columns from `minnie_features.parquet` intentionally not written:**
- `root_id`, `pt_root_id_y` — segment root ids, not feature values.
- `valence` — exc/inh label; written in a later cluster-membership notebook.
- `cell_type`, `classification_system`, `cell_id`, `pt_root_id_x` — cell type metadata; written in a later notebook.
- `core`, `group_old` — cohort membership flags; written in a later notebook.
- `pt_position_x/y/z` — voxel coordinates; superseded by the standard_transform micron coordinates written here.